In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

GoogleGeminiKey = os.getenv('GOOGLE_GEMINI_KEY')
connection = os.getenv("PG_CONNECTION_STRING")


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_postgres.vectorstores import PGVector

raw_documents = TextLoader("data/how_to_read_pnl.txt").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GoogleGeminiKey)
db = PGVector.from_documents(documents, embeddings, connection=connection)

In [14]:
# create retriever
retriever = db.as_retriever(search_kwargs={"k":10})

# fetch relevant documents.
docs = retriever.invoke("how to read pnl?")
docs

[Document(id='7f623b30-63c0-40a2-a295-a85513791a1c', metadata={'source': 'data/how_to_read_pnl.txt'}, page_content='Tata McGraw Hill Publishing Company Limited\nNEW DELHI\nN. Ramachandran\nPrincipal Consultant\nManagement Advisory Services\nKochi\nRam Kumar Kakani\nAssociate Professor, XLRI\nJamshedpur\nHow to Read\nA\nPROFIT AND LOSS\nSTATEMENT\nTata McGraw Hill Professional: Finance Made Easy Series\nPublished by Tata McGraw Hill Education Private Limited,\n7 West Patel Nagar, New Delhi 110 008.\nCopyright © 2010, by Tata McGraw Hill Education Private Limited\nNo part of this publication may be reproduced or distributed in any form or by any\nmeans, electronic, mechanical, photocopying, recording, or otherwise or stored in a\ndatabase or retrieval system without the prior written permission of the publishers. The\nprogram listings (if any) may be entered, stored and executed in a computer system,\nbut they may not be reproduced for publication.\nThis edition can be exported from Indi

In [23]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Markdown, display

llm = GoogleGenerativeAI(model="gemini-2.0-flash-001",api_key=GoogleGeminiKey,temperature=0)

prompt_template = ChatPromptTemplate.from_messages(
[("system", """
            Answer the question on the based on the given context below. If the question cannot be answered using the information
provided in the context, respond with "I don't know".
    """),
    ("system","Answer always in markdown format and List the items."),
    ("human", "Context: {context}"),
    ("human", "Question: {question}")])

chain = prompt_template | llm

query = "how to read pnl?"

docs = retriever.get_relevant_documents(query)

result = chain.invoke({"context": docs, "question": query})

display(Markdown(result))


To effectively read a Profit and Loss (P&L) statement, here are key steps and components to consider:

1.  **Understand the basic structure:**
    *   Sales (or Revenues)
    *   Less: Cost of Goods Sold
    *   = Gross Profit
    *   Less: Operating Expenses
    *   = Operating Profit
    *   Less: Non-Operating Expenses
    *   = Profit Before Interest and Tax (PBIT)
    *   Less: Interest
    *   = Profit Before Tax (PBT)
    *   Less: Tax
    *   = Profit After Tax (PAT)
    *   Add: Previous year’s balance of P&L A/C
    *   = Profit Available for Distribution
    *   Less: Appropriations
    *   = Retained Earnings

2.  **Key Profit Metrics:**
    *   **Gross Profit:**  Indicates the profit a company makes after deducting the costs associated with producing and selling its products or services.
    *   **Operating Profit:**  Reveals the profit from core business operations before interest and taxes.
    *   **PBIT (Profit Before Interest and Tax):**  Earnings before accounting for interest expenses and taxes.
    *   **PBT (Profit Before Tax):**  Earnings before taxes.
    *   **PAT (Profit After Tax):**  The net income earned by a company after providing for tax. It measures the net earnings available to the shareholders.

3.  **Profit Available for Distribution:**
    *   Sum up PAT and the retained earnings from previous years.
    *   This figure indicates the amount available for distribution, such as dividends.

4.  **Retained Earnings:**
    *   Derived by deducting the dividend to be paid to shareholders from the profit available for distribution.

5.  **Non-Operating Income:**
    *   Income from activities that are not part of the core business.

6.  **Depreciation Methods:**
    *   **Straight Line/Fixed Method:** A fixed percentage of the original cost of the asset is depreciated each year until the asset value becomes nil.
    *   **Written Down Value Method:** Also referred to as the remaining book value, this is the value at which assets are recorded on the balance sheet.

7.  **Importance of Income Statement:**
    *   Shows the change in owner's equity affected by retained earnings.
    *   Provides knowledge about the operations of the business.

In [25]:
from langchain_core.runnables import chain

@chain
def chat(input):
    # fetch relevant documents
    docs = retriever.get_relevant_documents(input)
    
    # formatted prompt 
    formatted = prompt_template.invoke({"context": docs, "question": input})

    # generate response
    answer = llm.invoke(formatted)

    return {"answer": answer, "docs": docs}


chat.invoke(query)

{'answer': "To effectively read a Profit and Loss (P&L) statement, here are key steps and components to consider:\n\n1.  **Sales (or revenues)**: This is the starting point, representing the total income generated from the company's primary activities.\n2.  **Cost of goods sold**: Deducting this from sales gives you the gross profit.\n3.  **Gross profit**: This is the profit earned before considering operating expenses.\n4.  **Operating expenses**: These are the costs incurred in running the business, such as administrative and selling expenses. Subtracting operating expenses from gross profit yields the operating profit.\n5.  **Operating profit**:  This reflects the profit from the company's core operations.\n6.  **Non-operating expenses**: These are expenses not directly related to the core business activities.\n7.  **Profit before interest and tax (PBIT)**: This is the profit before accounting for interest and taxes.\n8.  **Interest**: Deducting interest expenses leads to the profit

### Rewrite-Retrieve-Read

In [33]:
query = """
Today I woke up and realized that my laptop battery was low.then I found a charger in the kitchen but it wasn't working properly.
So I decided to ask for help from my friend who is an engineer. How to read PnL?
"""

res = chat.invoke(query)
print(res['answer'], res['docs'])

I don't know. [Document(id='46fda176-7371-4384-acda-9b85b009634c', metadata={'source': 'data/how_to_read_pnl.txt'}, page_content='How to Read\nA\nPROFIT AND LOSS\nSTATEMENT\nTata McGraw Hill Professional: Finance Made Easy Series\nTata McGraw Hill Professional: Finance Made Easy Series\nFinancial success is the raison d’être of any business, and fi nancial health\nof any organization is refl ected in its fi nancial statements. But, it has been\nobserved that managerial professionals often have little understanding of\nfi nance and little time to read treatises on it. Further, fi nancial statements\nare regarded as too complex to understand and left to be ‘deciphered’ by\nfi nance experts. Hence, cultivating a culture of awareness and transparency\nof fi nance is a prime imperative.\nFinance Made Easy Series has been designed to impart management\nexecutives with adequate knowledge to understand and appreciate fi nancial\nstatements and their implications for the fi scal solvency of the

In [50]:
rewrite_prompt = ChatPromptTemplate.from_template(
    """
    Provide a better search query for web search engine to answer the question below,
    end the queries with "**". 
    Question: {question}
    """
)

def parse_rewrite_output(message):
    return message.split('**')[0]

rewrite = rewrite_prompt | llm | parse_rewrite_output

# rewrite.invoke(query)

@chain
def chat_rrr(query):
    rrr_query = rewrite.invoke(query)
    return chat.invoke(rrr_query)

res = chat_rrr.invoke(query)
print(res['answer'], res['docs'])



Here are key points about understanding a Profit and Loss (P&L) statement for beginners, based on the provided context:

*   **Purpose**: A P&L statement, also known as an Income Statement, summarizes a company's revenues and expenses over a specific period to determine its profit or loss.

*   **Importance**: It reflects the financial health of a company and helps stakeholders understand the profitability, compare performance over time, and predict future profits.

*   **Basic Calculation**: The basic formula is Revenues - Expenses = Profit (if revenues exceed expenses) or Loss (if expenses exceed revenues).

*   **Key Components**:
    *   **Revenue**: The income generated from sales of goods or services.
    *   **Cost of Goods Sold**: Direct costs associated with producing goods or delivering services.
    *   **Gross Profit**: Revenue minus the cost of goods sold.
    *   **Operating Expenses**: Expenses incurred during normal business operations (e.g., personnel, depreciation).
 

In [51]:
@chain
def chat_rrr(query):
    new_query = rewrite.invoke(query)
    
    docs = retriever.get_relevant_documents(new_query)

    formatted_docs = prompt_template.invoke({"question": new_query, "context": docs})

    return llm.invoke(formatted_docs)

print(query)    
print(chat_rrr.invoke(query))



Today I woke up and realized that my laptop battery was low.then I found a charger in the kitchen but it wasn't working properly.
So I decided to ask for help from my friend who is an engineer. How to read PnL?

I don't know
